# N-sweep / D-sweep SAGD heatmap grid

Discovers every completed run under `scripts/run/data/` matching the `run_nsweep.sh` naming scheme (`{norm}_D{d}_n{n}/D{d}_N{n}_T{T}`) and plots a grid of SAGD distance-matrix heatmaps: **columns = dimension `d`**, **rows = `n_samples`**. Cells for a (d, n) combination that hasn't been run yet (or is still in progress) are left blank.

Currently `run_nsweep.sh` only sweeps `n_samples` at a fixed `D=1024`, so the grid will show a single populated column until more dimensions are run under the same naming scheme -- no notebook changes needed when that happens, just re-run it.

In [39]:
import re
from pathlib import Path

import numpy as np
import joblib
import matplotlib.pyplot as plt

from lib.ou_model import find_third_phase_onset
from lib.viz.plotting import _draw_sagd_heatmap_with_prob


## Discover runs

Parses `NORM` and `T` out of `run_nsweep.sh` (assumed fixed across the sweep), then finds every `{NORM}_D{d}_n{n}` experiment directory on disk and checks whether its pipeline outputs are complete.

In [40]:
NSWEEP_SCRIPT = Path("../scripts/run/run_nsweep.sh")
NSWEEP_DATA_ROOT = Path("../data")


def parse_bash_vars(script_path: Path) -> dict:
    """Pull simple `VAR=value` / `VAR="value"` assignments out of the sweep script."""
    text = script_path.read_text()
    values = dict(re.findall(r'^(\w+)=([0-9.]+)\s*$', text, flags=re.MULTILINE))
    values.update(re.findall(r'^(\w+)="([^"]*)"\s*$', text, flags=re.MULTILINE))
    return values


cfg = parse_bash_vars(NSWEEP_SCRIPT)
NORM = cfg["NORM"]
T = float(cfg["T"])

# history.jbl is deliberately excluded: it's ~200 MB per run and we no longer
# read it here (snapshot times come from CTDs.jbl, mu from the sweep script).
REQUIRED_FILES = ["Ws.jbl", "CTDs.jbl", "SAGD.jbl", "clusters.jbl"]

# `run_nsweep.sh` sweeps *all* dimensions in `DS` within a single experiment, so
# each experiment dir is named `{NORM}_D{DS}_n{n}` -- where `{DS}` is the whole
# space-separated dimension list -- and holds one `D{d}_N{n}_T{T}` subdirectory
# per dimension. So we match the experiment dir by `n` only and read the
# individual dimensions off the inner run subdirectories.
EXP_PATTERN = re.compile(rf"^{re.escape(NORM)}_D.*_n(\d+)$")
RUN_PATTERN = re.compile(rf"^D(\d+)_N(\d+)_T{int(T)}$")

run_dirs = {}  # (d, n_samples) -> run_dir
for exp_dir in sorted(p for p in NSWEEP_DATA_ROOT.iterdir() if p.is_dir()):
    m = EXP_PATTERN.match(exp_dir.name)
    if not m:
        continue
    n = int(m.group(1))
    for run_dir in sorted(p for p in exp_dir.iterdir() if p.is_dir()):
        rm = RUN_PATTERN.match(run_dir.name)
        if not rm:
            continue
        d, n_dir = int(rm.group(1)), int(rm.group(2))
        assert n_dir == n, f"n mismatch: {run_dir} has N={n_dir} under n{n} experiment"
        missing = [f for f in REQUIRED_FILES if not (run_dir / f).exists()]
        if missing:
            print(f"Skipping D={d}, n_samples={n}: run not finished yet (missing {missing}) at {run_dir}")
            continue
        run_dirs[(d, n)] = run_dir

d_list = sorted({d for d, _ in run_dirs})
n_list = sorted({n for _, n in run_dirs})
print(f"Discovered grid: dimensions={d_list}, n_samples={n_list}")


Discovered grid: dimensions=[256, 1024, 4096], n_samples=[1000, 2000, 4000]


## Load data

One entry per completed `(d, n_samples)` combination; missing combinations simply aren't keys in `grid_data` and are left blank in the plot.

In [41]:
grid_data = {}
std = 1.0  # bimodal_gaussian std used throughout the sweep
mu = float(cfg["MU"])  # constant across the sweep (read from run_nsweep.sh)

for (d, n), run_dir in run_dirs.items():
    breakpoints = joblib.load(run_dir / "clusters.jbl")
    ctds = joblib.load(run_dir / "CTDs.jbl")
    # Snapshot times are stored in the (lightweight) CTDs params as `ts`, so we
    # avoid deserializing the ~200 MB history.jbl just to read them.
    snaps = np.asarray(ctds["params"]["ts"])

    grid_data[(d, n)] = dict(
        W=joblib.load(run_dir / "SAGD.jbl"),
        time_snaps=snaps,
        ts=None,
        tsagd=snaps[breakpoints[0]],
        tstar=find_third_phase_onset(ctds["CTDs"], snaps),
        ctds=ctds,
        mu=mu,
    )

print(f"Loaded {len(grid_data)} / {len(run_dirs)} (d, n_samples) combinations")


Loaded 9 / 9 (d, n_samples) combinations


## Grid plot

Reuses `lib.viz.plotting._draw_sagd_heatmap_with_prob` per cell instead of redrawing the heatmap here. That helper ties its title and the theoretical same-cluster probability curve `φ(t)` to the same `d` argument -- which is exactly right in this grid, since `d` genuinely varies by column (unlike the earlier n_samples-only row plot, where `d` was fixed and had to be decoupled from the label).

In [42]:
def plot_sagd_heatmap_grid(
        grid_data, d_list, n_list, std,
        distance='SAGD', show_prob=True, show_legend=True,
        clipping=True, save_fig_path=None,
):
    """
    Grid of SAGD-distance heatmaps: columns = dimension `d`, rows = `n_samples`.
    `grid_data` maps (d, n_samples) -> dict with keys W, time_snaps, ts,
    tsagd, tstar, ctds, mu. Missing (d, n_samples) combinations are left blank.
    """
    if not d_list or not n_list:
        raise ValueError("d_list / n_list is empty -- no completed runs to plot.")

    nrows, ncols = len(n_list), len(d_list)
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(6 * ncols, 9.5 * nrows),
        gridspec_kw={'wspace': 0.5, 'hspace': 0.6},
        squeeze=False,
    )

    for row, n in enumerate(n_list):
        for col, d in enumerate(d_list):
            ax = axes[row, col]
            cell = grid_data.get((d, n))
            if cell is None:
                ax.axis('off')
                ax.text(0.5, 0.5, f"D={d}, N={n}\n(no data)",
                        ha='center', va='center', fontsize=11, color='gray',
                        transform=ax.transAxes)
                continue

            _draw_sagd_heatmap_with_prob(
                ax_hm=ax,
                W=cell["W"],
                time_vector=cell["time_snaps"],
                d=d,
                mu=cell["mu"],
                std=std,
                ts=cell["ts"],
                tsagd=cell["tsagd"],
                t_star=cell["tstar"],
                show_ylabel=(col == 0),
                show_legend=show_legend,
                show_prob=show_prob,
                distance=distance,
                ctds=cell["ctds"],
                model='synthetic',
                clipping=clipping,
            )
            if col == 0:
                ax.set_ylabel(f"N={n}\nTime", fontsize=12)

    if save_fig_path:
        Path(save_fig_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_fig_path, dpi=300, bbox_inches='tight')
    plt.show()


## Plot

In [43]:
# plot_sagd_heatmap_grid(
#     grid_data=grid_data,
#     d_list=d_list,
#     n_list=n_list,
#     std=std,
#     distance='SAGD',
#     show_prob=False,
#     save_fig_path='figs/sagd_heatmap_grid_d_x_nsamples.png',
# )


## Animated distributions per time step

For a chosen `(d, n_samples)`, animate how three per-snapshot distributions evolve along the
diffusion trajectory (`t = T -> 0`):

1. **Raw CTD** — the un-normalized Commute-Time-Distance distribution over all node pairs
   (`CTDs.jbl -> ctds`). Spans many orders of magnitude, so it's drawn on a **log x-axis**.
2. **Normalized CTD** — the same distribution after `--norm_type` normalization
   (`CTDs.jbl -> norm_ctds`); this is what the SAGD Wasserstein distance actually compares.
3. **Euclidean distance** — the distribution of pairwise Euclidean distances between the actual
   sample positions at that snapshot (computed from `history.jbl`).

The raw/normalized CTDs come straight from the cached `CTDs.jbl` (already compressed to
`n_bins` equal-frequency quantiles unless the run used `--keep_raw_ctds`). The Euclidean
distances aren't cached by the pipeline, so we compute them once from the (large) `history.jbl`
and cache them next to the other outputs as `euclid_dists.jbl` — subsequent runs read only the
small cache. Histogramming an equal-frequency-quantile array recovers the distribution shape,
so all three panels are directly comparable.

**Axes rescale per time step.** All three panels take their x-range from the central `coverage`
fraction (default `0.99`) of *that snapshot's* values, and their y-range from that frame's own
histogram. Fixed global limits don't work here: the distributions move by orders of magnitude
along the trajectory, so a shared x-window squeezes most frames into a few bins and a shared
y-limit (set by the single most peaked snapshot) flattens the CTD panels to invisibility. The
trade-off is that bar positions are no longer comparable across frames — read the tick labels.
Pass `euclid_xlim=(lo, hi)` to pin the Euclidean panel to a fixed window instead; that window is
dimension-dependent, since pairwise distances in `d` dimensions concentrate around `sqrt(2*d)`.

Histograms use `bins=200` bins *within* the per-frame window. The cached CTD arrays hold only
`n_bins` (5000) quantile points each, so pushing much past ~5000 bins leaves most bins empty by
construction -- higher resolution needs the pipeline re-run with a larger `--n_bins`.

The player exposes a **frame slider plus play/step buttons** (`FuncAnimation.to_jshtml`), so the
time step can be scrubbed manually or played through — no extra widget dependency required.

In [44]:
from scipy.spatial.distance import pdist
from joblib import Parallel, delayed

from lib.stats import compress_equal_frequency

EUCLID_FILE = "euclid_dists.jbl"


def _snapshot_euclid_quantiles(points: np.ndarray, n_bins: int) -> np.ndarray:
    """All pairwise Euclidean distances at one snapshot, compressed to `n_bins`
    equal-frequency quantiles (same scheme the pipeline uses for CTDs)."""
    return compress_equal_frequency(pdist(np.asarray(points), metric="euclidean"), n_bins)


def get_euclidean_quantiles(run_dir: Path, n_bins: int = 5000, n_jobs: int = 16) -> dict:
    """Per-snapshot pairwise-Euclidean-distance distributions for one run.

    Cached as `euclid_dists.jbl` next to the other pipeline outputs, so the
    large `history.jbl` is deserialized only the first time. Returns a dict
    mapping snapshot time -> compressed quantile array, ordered T -> 0.
    """
    cache = run_dir / EUCLID_FILE
    if cache.exists():
        return joblib.load(cache)["euclid"]

    hist = joblib.load(run_dir / "history.jbl")["history"]  # {t: (N, d) array}
    ts = list(hist.keys())
    dists = Parallel(n_jobs=n_jobs)(
        delayed(_snapshot_euclid_quantiles)(hist[t], n_bins) for t in ts
    )
    euclid = dict(zip(ts, dists))
    joblib.dump(
        {"euclid": euclid, "params": {"ts": ts, "n_bins": n_bins}},
        cache,
        compress=3,
    )
    return euclid


PROJ_FILE = "proj2d.jbl"
PROJ_VERSION = 2


def get_projection_and_cut(run_dir: Path, basis: np.ndarray) -> dict:
    """Per-snapshot 2-D projection of the samples plus the kNN between-cluster cut.

    A point's cluster is the centre it is closer to, which for the bimodal
    mixture -- equal weights, one shared isotropic variance -- is exactly the
    sign of its coordinate along the mean-difference direction, i.e. of the
    first projected coordinate (the same rule as `ou_model.classify`).

    Returns a dict mapping snapshot time to `xy` (N, 2) projected samples,
    `labels` (N,) in {-1, +1}, `edges` (2, E) / `edge_w` (E,) for the graph
    edges joining the two clusters, `cut`, their total weight, and
    `n_edges` / `weight` for the whole graph, so the cut can be reported as a
    fraction of either.

    Cached as `proj2d.jbl` next to the other pipeline outputs, since building it
    reads both `history.jbl` and the large `Ws.jbl` once.
    """
    cache = run_dir / PROJ_FILE
    if cache.exists():
        stored = joblib.load(cache)
        if (stored["params"].get("version") == PROJ_VERSION
                and np.allclose(stored["params"]["basis"], basis)):
            return stored["proj"]

    hist = joblib.load(run_dir / "history.jbl")["history"]
    ts = list(hist.keys())
    xys = [np.asarray(hist[t], dtype=float) @ basis for t in ts]
    del hist

    graphs = joblib.load(run_dir / "Ws.jbl")
    assert np.allclose(graphs["ts"], ts), "graph / history snapshot times differ"

    proj = {}
    for t, xy, W in zip(ts, xys, graphs["Ws"]):
        labels = np.where(xy[:, 0] >= 0, 1, -1).astype(np.int8)
        i, j = np.triu(W, 1).nonzero()
        w = W[i, j]
        between = labels[i] != labels[j]
        proj[t] = {
            "xy": xy.astype(np.float32),
            "labels": labels,
            "edges": np.stack([i[between], j[between]]).astype(np.int32),
            "edge_w": w[between].astype(np.float32),
            "cut": float(w[between].sum()),
            "n_edges": int(i.size),
            "weight": float(w.sum()),
        }
    del graphs

    joblib.dump(
        {"proj": proj, "params": {"ts": ts, "basis": basis, "version": PROJ_VERSION}},
        cache,
        compress=3,
    )
    return proj


In [55]:
import matplotlib.animation as animation
from IPython.display import HTML
from matplotlib.collections import LineCollection

from lib.ou_model import mixture_at_time

# (title, key, x-axis color, log-x?) for the two distribution panels sitting to
# the right of the mixture-contour panel.
_PANELS = [
    ("Normalized CTD",     "norm_ctds", "seagreen",  False),
    ("Euclidean distance", "euclid",    "indianred", False),
]


def _mixture_plane_basis(mu_star, seed=0):
    """(d, 2) orthonormal basis of the plane the bimodal mixture is drawn in.

    Column 1 is the unit vector along the mean difference mu_+ - mu_-, the only
    direction carrying cluster information; column 2 is an arbitrary orthogonal
    direction, along which both components have identical marginals.

    An orthogonal projection maps N(mu, Delta*I_d) onto N(B.T @ mu, Delta*I_2)
    exactly, so -- the mean difference lying inside the plane -- the drawn 2-D
    mixture has the same component overlap mass as the d-dimensional one, not
    an approximation of it. The basis is time-independent: `mixture_at_time`
    only rescales the means by exp(-t), it never rotates them.
    """
    mu = np.asarray(mu_star, dtype=float)
    u = mu / np.linalg.norm(mu)
    rng = np.random.default_rng(seed)
    v = rng.standard_normal(u.size)
    v -= (v @ u) * u
    v /= np.linalg.norm(v)
    return np.stack([u, v], axis=1)


def _draw_mixture_panel(ax, t, mu_star, std, basis, snap,
                        radii=(1.0, 2.0, 3.0), grid=401, max_edges=20):
    """Bimodal mixture at time t drawn in the mean-difference plane, in units of
    the (shared) component standard deviation sigma(t).

    Contour lines of the exact 2-D marginal sit `radii` standard deviations out
    from a component centre, so each blob keeps a constant apparent width while
    the centres travel inward as exp(-t); the window is rescaled per frame to
    +-(delta/2 + 4), like the histogram panels. On top go the projected samples,
    coloured by cluster, and the kNN edges that join the two clusters (at most
    `max_edges` of them are drawn, evenly spaced through the list, but the
    reported cut counts and weights are always the full sums, each shown against
    the whole graph's total). `snap` is one time slice of
    `get_projection_and_cut`.
    """
    means, variances, w = mixture_at_time(t, mu_star, std)
    assert np.allclose(variances, variances[0]), \
        "panel assumes one shared isotropic variance across components"
    sigma = float(np.sqrt(variances[0]))
    c = (means @ basis) / sigma
    lim = np.abs(c).max() + 4.0

    g = np.linspace(-lim, lim, grid)
    Z1, Z2 = np.meshgrid(g, g)
    dens = np.zeros_like(Z1)
    for (c1, c2), wk in zip(c, w):
        dens += wk * np.exp(-0.5 * ((Z1 - c1)**2 + (Z2 - c2)**2)) / (2 * np.pi)

    peak = w.max() / (2 * np.pi)
    levels = [peak * np.exp(-0.5 * r**2) for r in sorted(radii, reverse=True)]
    top = max(dens.max(), levels[-1]) * 1.01
    ax.contourf(Z1, Z2, dens, levels=levels + [top], cmap="Blues", alpha=0.55)
    ax.contour(Z1, Z2, dens, levels=levels, colors="steelblue", linewidths=1.0)

    z = snap["xy"] / sigma
    pos = snap["labels"] > 0
    ax.scatter(z[pos, 0], z[pos, 1], s=2, c="0.25", alpha=0.30, linewidths=0)
    ax.scatter(z[~pos, 0], z[~pos, 1], s=2, c="0.55", alpha=0.30, linewidths=0)

    i, j = snap["edges"]
    if i.size:
        step = max(1, int(np.ceil(i.size / max_edges)))
        segs = np.stack([z[i[::step]], z[j[::step]]], axis=1)
        alpha = float(np.clip(400 / len(segs), 0.06, 0.6))
        ax.add_collection(LineCollection(segs, colors="crimson", linewidths=0.6,
                                         alpha=alpha, zorder=3))
    ax.plot(c[:, 0], c[:, 1], "kx", ms=6, mew=1.5, zorder=4)

    delta = float(np.linalg.norm(c[0] - c[1]))
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_title("Gaussian mixture (mean-difference plane)", fontsize=11)
    ax.set_xlabel(r"along $(\mu_+-\mu_-)/\|\cdot\|$")
    ax.set_ylabel(r"orthogonal direction")
    ax.text(0.02, 0.98,
            f"$\\delta$ = {delta:.3g} $\\sigma$\n"
            f"cut edges  = {i.size:d} / {snap['n_edges']:d}"
            f"  ({i.size / snap['n_edges']:.1%})\n"
            f"cut weight = {snap['cut']:.4g} / {snap['weight']:.5g}"
            f"  ({snap['cut'] / snap['weight']:.1%})",
            transform=ax.transAxes, va="top", ha="left", fontsize=8)


def _panel_edges(dist, bins, logx, coverage=0.99, xlim=None):
    """Histogram bin edges for a *single* snapshot's distribution, spanning the
    central `coverage` fraction of its values (so ~99% of the observations fall
    inside the axis at that time step). `xlim` pins the range explicitly
    instead of deriving it from the data."""
    if xlim is not None:
        lo, hi = xlim
    else:
        tail = (1.0 - coverage) / 2
        lo, hi = np.quantile(dist, [tail, 1.0 - tail])
    if hi <= lo:                                       # degenerate snapshot
        hi = lo + (abs(lo) * 1e-6 or 1e-12)
    if logx:
        lo = max(lo, 1e-12)
        hi = max(hi, lo * (1 + 1e-9))
        return np.logspace(np.log10(lo), np.log10(hi), bins + 1)
    return np.linspace(lo, hi, bins + 1)


def animate_distributions(d, n, stride=2, bins=200, coverage=0.99,
                          t_range=(1.0, 3.0), euclid_xlim=None, n_jobs=16,
                          fps=6, save_path=None):
    """Animated Gaussian-mixture contours / normalized-CTD / Euclidean-distance
    panels for the completed run `(d, n_samples)`, one frame per snapshot time.

    The left panel is the analytic bimodal mixture at that time
    (`ou_model.mixture_at_time`), projected onto the plane spanned by the mean
    difference and one orthogonal direction (see `_mixture_plane_basis`) -- an
    exact 2-D marginal, so its component overlap is the d-dimensional one --
    overlaid with the run's own samples in the same projection and the kNN-graph
    edges that join the two clusters. The two right panels are the empirical
    distance distributions of the run.

    Returns an ``IPython.display.HTML`` player (frame slider + play/step
    controls) so the time step can be scrubbed manually or played through.
    `stride` sub-samples snapshots to keep the embedded animation small.

    Each panel is rescaled *per time step*: the x-range covers the central
    `coverage` fraction (default 99%) of that snapshot's values, and the y-range
    is set from that frame's own histogram -- so a single high-count snapshot no
    longer flattens the rest of the animation. The distributions shift by orders
    of magnitude along the trajectory, hence the per-frame limits; watch the
    tick labels, not the bar positions, when comparing frames.
    `euclid_xlim=(lo, hi)` pins the Euclidean panel to a fixed window instead
    (the useful window scales with `d`, since pairwise distances in d dimensions
    concentrate around sqrt(2*d)).

    `t_range=(t_lo, t_hi)` truncates the animation to snapshots with
    `t_lo <= t <= t_hi` (default (1.0, 3.0), the window around the transition);
    pass `t_range=None` to animate the whole trajectory. The per-frame axis
    limits are computed from the retained snapshots only.

    If `save_path` is given, the animation is also written to disk (parent dirs
    created as needed). The format follows the extension:
      * `.gif`          -> rendered frames via the pillow writer;
      * `.html`/`.htm`  -> the self-contained scrubbable jshtml player.
    (`.mp4` needs ffmpeg, which isn't installed here -- use `.gif`.)
    """
    if (d, n) not in grid_data:
        raise KeyError(
            f"No completed run for (d={d}, n_samples={n}). "
            f"Available: {sorted(grid_data)}"
        )

    cell = grid_data[(d, n)]
    ctds = cell["ctds"]["CTDs"]                       # {t: {'ctds', 'norm_ctds'}}
    euclid = get_euclidean_quantiles(run_dirs[(d, n)], n_jobs=n_jobs)
    snaps = list(ctds.keys())                          # ordered T -> 0
    assert list(euclid.keys()) == snaps, "CTD / Euclidean snapshot times differ"
    if t_range is not None:
        t_lo, t_hi = t_range
        snaps = [t for t in snaps if t_lo <= t <= t_hi]
        if not snaps:
            raise ValueError(
                f"No snapshots inside t_range={t_range}; available times span "
                f"[{min(ctds):.3g}, {max(ctds):.3g}]"
            )
    tsagd, tstar = cell["tsagd"], cell["tstar"]

    # Pre-materialize each panel's per-snapshot distribution + per-snapshot
    # bin edges / y-limit.
    sources = {"norm_ctds": lambda t: np.asarray(ctds[t]["norm_ctds"]),
               "euclid": lambda t: np.asarray(euclid[t])}
    mu_star = np.full(d, cell["mu"])
    basis = _mixture_plane_basis(mu_star)
    proj = get_projection_and_cut(run_dirs[(d, n)], basis)
    panel_xlim = {"euclid": euclid_xlim}
    panel_data, panel_edges, panel_ymax = [], [], []
    for _title, key, _c, logx in _PANELS:
        dists = [sources[key](t) for t in snaps]
        edges = [_panel_edges(a, bins, logx, coverage, panel_xlim.get(key))
                 for a in dists]
        ymax = [np.histogram(a, bins=e)[0].max() for a, e in zip(dists, edges)]
        if max(ymax) == 0:
            print(f"warning: no {key} values fall inside {panel_xlim.get(key)} -- "
                  f"panel will be empty (try a wider euclid_xlim for d={d})")
        panel_data.append(dists)
        panel_edges.append(edges)
        panel_ymax.append([max(m, 1) * 1.08 for m in ymax])

    frames = list(range(0, len(snaps), stride))
    if frames[-1] != len(snaps) - 1:
        frames.append(len(snaps) - 1)                  # always end at t -> 0

    fig = plt.figure(figsize=(16, 5))
    gs = fig.add_gridspec(2, 3, height_ratios=[1, 9], hspace=0.55, wspace=0.28)
    ax_time = fig.add_subplot(gs[0, :])
    axes = [fig.add_subplot(gs[1, c]) for c in range(3)]

    # Static timeline strip: regime boundaries + a moving "current time" marker.
    T_max, T_min = max(snaps), min(snaps)
    pad = 0.02 * (T_max - T_min) or 1e-3
    ax_time.set_xlim(max(0.0, T_min - pad), T_max + pad)
    ax_time.set_ylim(0, 1)
    ax_time.set_yticks([])
    ax_time.set_xlabel("diffusion time  t  (T → 0)")
    ax_time.axvline(tsagd, color="darkcyan", lw=2, label=fr"$t_{{sagd}}$={tsagd:.2f}")
    ax_time.axvline(tstar, color="darkorange", lw=2, ls="--", label=fr"$t^*$={tstar:.2f}")
    ax_time.legend(loc="center left", bbox_to_anchor=(1.005, 0.5),
                   fontsize=8, frameon=False)
    time_marker = ax_time.axvline(T_max, color="red", lw=2.5)

    def draw(frame_idx):
        t = snaps[frame_idx]
        time_marker.set_xdata([t, t])
        axes[0].clear()
        _draw_mixture_panel(axes[0], t, mu_star, std, basis, proj[t])
        for ax, dists, edges, ymax, (title, _key, color, logx) in zip(
                axes[1:], panel_data, panel_edges, panel_ymax, _PANELS):
            e = edges[frame_idx]
            ax.clear()
            ax.hist(dists[frame_idx], bins=e, color=color, alpha=0.85)
            if logx:
                ax.set_xscale("log")
            ax.set_xlim(e[0], e[-1])
            ax.set_ylim(0, ymax[frame_idx])
            ax.set_title(title, fontsize=11)
            ax.set_xlabel("distance")
            ax.set_ylabel("count (quantile bins)")
        regime = ("noise" if t > tsagd else
                  "SAGD onset" if t > tstar else "resolved clusters")
        fig.suptitle(f"D={d}, N={n}   |   t = {t:.2f}   |   regime: {regime}",
                     fontsize=13, y=1.0)

    anim = animation.FuncAnimation(
        fig, lambda k: draw(frames[k]), frames=len(frames), interval=1000 / fps
    )

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        suffix = save_path.suffix.lower()
        if suffix == ".gif":
            anim.save(save_path, writer="pillow", fps=fps)
        elif suffix in (".html", ".htm"):
            save_path.write_text(anim.to_jshtml(fps=fps))
        else:
            plt.close(fig)
            raise ValueError(
                f"Unsupported save extension '{suffix}'. Use '.gif' or '.html' "
                "('.mp4' requires ffmpeg, which isn't installed)."
            )
        print(f"Saved animation -> {save_path}")

    html = HTML(anim.to_jshtml(fps=fps))
    plt.close(fig)
    return html


In [56]:
# Pick any completed (d, n_samples); first call builds & caches euclid_dists.jbl
# for that run (reads the large history.jbl once), later calls are fast.
print("available (d, n_samples):", sorted(grid_data))

# `save_path` also writes the animation to disk: '.gif' (rendered frames) or
# '.html' (self-contained scrubbable player). Both land under figs/.
animate_distributions(d=1024, n=4000, 
                      save_path="figs/dist_animation_D256_N1000.html", t_range=(0.5, 5.0))


available (d, n_samples): [(256, 1000), (256, 2000), (256, 4000), (1024, 1000), (1024, 2000), (1024, 4000), (4096, 1000), (4096, 2000), (4096, 4000)]


Saved animation -> figs/dist_animation_D256_N1000.html


## Cut weight vs. CTD peak distance

Between-cluster cut weight fraction (left axis) against the gap between the two highest peaks of
the normalized CTD distribution (right axis) -- both ask when the graph stops mixing the two
clusters, so the point is whether they turn over together. `lib.viz.cut_peaks` holds the plot and
the peak-finding heuristic, shared with `ctd_investigation.ipynb`; the run is passed in as its
loaded `grid_data` entry plus its output directory.

In [ ]:
from lib.viz.cut_peaks import plot_cut_and_ctd_peaks

D_CUT, N_CUT = 1024, 4000
plot_cut_and_ctd_peaks(
    grid_data[(D_CUT, N_CUT)], run_dirs[(D_CUT, N_CUT)],
    d=D_CUT, n=N_CUT, std=std, t_range=(1.0, 3.0),
    save_fig_path=f"figs/cut_vs_ctd_peaks_D{D_CUT}_N{N_CUT}_t1_3.png",
);